In [ ]:
import pandas as pd

df = pd.read_csv('filtered-data/muestra_ratings.csv')

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

from surprise import Dataset, Reader, KNNBasic

config = {
    'test_size': 0.2,
    'random_state': 42,
    'k_neighbors': 40,
    'min_k': 1,
}

train_df, test_df = train_test_split(
    df[['userId', 'movieId', 'rating']].copy(),
    test_size=config['test_size'],
    random_state=config['random_state']
 )

def _metrics_from_preds(y_true_arr, y_pred_arr, n_test):
    rmse_val = float(np.sqrt(np.mean((y_true_arr - y_pred_arr) ** 2))) if len(y_true_arr) else float('nan')
    mae_val = float(np.mean(np.abs(y_true_arr - y_pred_arr))) if len(y_true_arr) else float('nan')
    coverage_val = float(len(y_pred_arr) / n_test) if n_test else 0.0
    return rmse_val, mae_val, coverage_val

y_true = np.array(test_df['rating'].astype(float).values, dtype=np.float32)
trained_models = {}
metrics = []

reader = Reader(rating_scale=(0.5, 5.0))
train_data = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader)
trainset = train_data.build_full_trainset()

surprise_specs = [
    ('user_user_cosine', {'name': 'cosine', 'user_based': True}),
    ('user_user_pearson', {'name': 'pearson', 'user_based': True}),
    ('item_item_cosine', {'name': 'cosine', 'user_based': False}),
    ('item_item_pearson', {'name': 'pearson', 'user_based': False}),
]

for model_name, sim_options in surprise_specs:
    algo_i = KNNBasic(
        k=config['k_neighbors'],
        min_k=config['min_k'],
        sim_options=sim_options,
        verbose=False
    )
    algo_i.fit(trainset)

    y_pred_i = np.array([
        float(algo_i.predict(uid=row.userId, iid=row.movieId, r_ui=row.rating).est)
        for row in test_df[['userId', 'movieId', 'rating']].itertuples(index=False)
    ], dtype=np.float32)

    rmse_i, mae_i, coverage_i = _metrics_from_preds(y_true, y_pred_i, len(test_df))

    trained_models[model_name] = algo_i
    metrics.append({
        'model_name': model_name,
        'user_based': bool(sim_options['user_based']),
        'similarity': sim_options['name'],
        'rmse': rmse_i,
        'mae': mae_i,
        'coverage': coverage_i,
    })

metrics_df = pd.DataFrame(metrics).sort_values(['user_based', 'similarity', 'model_name']).reset_index(drop=True)
metrics_df

## Exportar modelos
Guardar los 4 modelos entrenados (cosine/pearson, user-user/item-item) y su metadata.

In [ ]:
from pathlib import Path
from surprise import dump
import json

model_dir = Path('artifacts')
model_dir.mkdir(parents=True, exist_ok=True)

export_rows = []
for model_name, model_obj in trained_models.items():
    model_path = model_dir / f'model_{model_name}.pkl'
    metadata_path = model_dir / f'model_{model_name}_metadata.json'

    dump.dump(str(model_path), algo=model_obj)

    row = metrics_df.loc[metrics_df['model_name'] == model_name].iloc[0]
    metadata = {
        'model_name': model_name,
        'backend': 'surprise',
        'algorithm': 'KNNBasic',
        'library': 'surprise',
        'similarity': str(row['similarity']),
        'user_based': bool(row['user_based']),
        'k_neighbors': int(config['k_neighbors']),
        'min_k': int(config['min_k']),
        'train_users': int(train_df['userId'].nunique()),
        'train_items': int(train_df['movieId'].nunique()),
        'train_ratings': int(len(train_df)),
        'test_ratings': int(len(test_df)),
        'rmse': float(row['rmse']),
        'mae': float(row['mae']),
        'coverage': float(row['coverage'])
    }

    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    export_rows.append({
        'model_name': model_name,
        'backend': 'surprise',
        'model_path': str(model_path),
        'metadata_path': str(metadata_path)
    })

export_df = pd.DataFrame(export_rows).sort_values('model_name').reset_index(drop=True)
export_df